In [1]:
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # %reload_ext autotime  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution
import geopandas as gpd
import pandas as pd
from glob import glob
from sklearn.linear_model import LinearRegression
from tqdm.auto import tqdm
from tqdm.contrib.concurrent import process_map
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
from coastsat import SDS_transects
pd.options.plotting.backend = "plotly"

ValueError: Could not find plotting backend 'plotly'. Ensure that you've installed the package providing the 'plotly' entrypoint, or that the package has a top-level `.plot` method.

In [2]:
# Transects, origin is landward. Has beach_slope
transects = gpd.read_file("transects_extended.geojson")
transects.set_index("id", inplace=True)
transects

,site_id,orientation,along_dist,along_dist_norm,beach_slope,cil,ciu,trend,n_points,n_points_nonan,r2_score,mae,mse,rmse,intercept,ERODIBILITY,geometry
id,,,,,,,,,,,,,,,,,
aus0001-0000,aus0001,104.347648,0.000000,0.000000,0.085,0.0545,0.2000,-1.441081,767.0,428.0,0.168420,28.102591,1263.560863,35.546601,179.085729,None,"LINESTRING (153.26555 -24.7007, 153.26938 -24...."
aus0001-0001,aus0001,93.495734,98.408334,0.002935,0.050,0.0387,0.0640,-1.037105,767.0,569.0,0.097874,25.419324,1033.770813,32.152306,212.247788,None,"LINESTRING (153.26525 -24.7019, 153.2692 -24.7..."
aus0001-0002,aus0001,82.069341,198.408334,0.005918,0.050,0.0428,0.0647,-0.680019,767.0,588.0,0.053927,22.632907,838.007507,28.948359,205.106151,None,"LINESTRING (153.26539 -24.70316, 153.26931 -24..."
aus0001-0003,aus0001,81.192757,298.402523,0.008900,0.055,0.0480,0.0659,-0.405198,767.0,598.0,0.023412,20.749758,698.653187,26.432048,191.745881,None,"LINESTRING (153.26555 -24.70408, 153.26945 -24..."
aus0001-0004,aus0001,81.065473,398.402523,0.011882,0.075,0.0614,0.0922,-0.090025,767.0,608.0,0.001277,19.889328,655.810616,25.608800,175.092121,None,"LINESTRING (153.2657 -24.70497, 153.26961 -24...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ber0002-0009,ber0002,NaN,NaN,NaN,0.080,0.0638,0.1065,0.159612,199.0,199.0,0.041461,4.326802,33.378983,5.777455,127.283966,None,"LINESTRING (-64.82204 32.25336, -64.82017 32.2..."
ber0002-0010,ber0002,NaN,NaN,NaN,0.085,0.0673,0.1096,0.071946,199.0,197.0,0.010730,4.357300,26.732016,5.170301,128.858980,None,"LINESTRING (-64.82143 32.25361, -64.82029 32.2..."
ber0002-0011,ber0002,NaN,NaN,NaN,0.105,0.0797,0.1462,0.081426,199.0,198.0,0.011823,4.779105,31.534469,5.615556,129.347401,None,"LINESTRING (-64.82118 32.25369, -64.82004 32.2..."


In [3]:
vos_files = pd.Series(
    sorted(glob("csv_run7/*/time_series_tidally_corrected.csv"))
)
vos_files = vos_files[~vos_files.str.contains("nzd")]
vos_files

Series([], dtype: object)

In [4]:
my_files = pd.Series(
    sorted(glob("data/*/transect_time_series_tidally_corrected.csv"))
)
my_files

0    data/nzd0001/transect_time_series_tidally_corr...
1    data/sar0001/transect_time_series_tidally_corr...
dtype: object

In [5]:
sar_files = pd.Series(sorted(glob("data/sar*/transect_time_series.csv")))
sar_files

0    data/sar0001/transect_time_series.csv
dtype: object

In [6]:
files = pd.concat([vos_files, my_files, sar_files])
files

0    data/nzd0001/transect_time_series_tidally_corr...
1    data/sar0001/transect_time_series_tidally_corr...
0                data/sar0001/transect_time_series.csv
dtype: object

In [7]:
f = files[files.str.contains("ber0001")].iloc[0]
# despiked_filename = f.replace(".csv", "_tidally_corrected.csv")
df = pd.read_csv(f)
df.dates = pd.to_datetime(df.dates)
df.set_index("dates", inplace=True)
display(df.columns)
import matplotlib.pyplot as plt

transect_id = "ber0001-0002"


def custom_mean(window):
    return window[window.between(window.quantile(0.25), window.quantile(0.75))].mean()


pd.DataFrame(
    {
        "raw": df[transect_id],
        "rolling 90d mean": df[transect_id].rolling("90d", min_periods=1).mean(),
        "rolling 180d mean": df[transect_id].rolling("180d", min_periods=1).mean(),
        "rolling 90d custom mean": df[transect_id]
        .rolling("90d", min_periods=1)
        .apply(custom_mean),
        "rolling 180d custom mean": df[transect_id]
        .rolling("180d", min_periods=1)
        .apply(custom_mean),
        # "rolling 365d": df[transect_id].rolling("365d", min_periods=1).mean(),
    },
    index=df.index,
).plot()

IndexError: single positional indexer is out-of-bounds

In [8]:
df = pd.read_csv("data/sar0939/transect_time_series.csv")
df.dates = pd.to_datetime(df.dates)
df.set_index("dates", inplace=True)
(df["sar0939-0000"] - 93).plot()

FileNotFoundError: [Errno 2] No such file or directory: 'data/sar0939/transect_time_series.csv'

In [9]:
def despike(chainage, threshold=40):
    chainage = chainage.dropna()
    chainage, dates = SDS_transects.identify_outliers(
        chainage.tolist(), chainage.index.tolist(), threshold
    )
    return pd.Series(chainage, index=dates)


def get_trends(f):
    df = pd.read_csv(f)
    try:
        df.dates = pd.to_datetime(df.dates)
    except:
        print(f)
    if "sar" in f or "ber" in f:
        smoothed_filename = f.replace(".csv", "_smoothed.csv")
        try:
            df = pd.read_csv(smoothed_filename)
            df.dates = pd.to_datetime(df.dates)
        except:
            df.dates = pd.to_datetime(df.dates)
            df.set_index("dates", inplace=True)
            satname = df.satname
            df = df.drop(columns="satname").apply(despike, axis=0)
            df["satname"] = satname
            df.reset_index(names="dates").to_csv(
                f.replace(".csv", "_despiked.csv"), index=False
            )
            for transect_id in df.drop(columns="satname").columns:
                df[transect_id] = df[transect_id].rolling("180d", min_periods=1).mean()
            df.reset_index(names="dates", inplace=True)
            df.to_csv(f.replace(".csv", "_smoothed.csv"), index=False)
    df.index = (df.dates - df.dates.min()).dt.days / 365.25
    df.drop(columns=["dates", "satname", "Unnamed: 0"], inplace=True, errors="ignore")
    trends = []
    for transect_id in df.columns:
        sub_df = df[transect_id].dropna()
        if not len(sub_df):
            continue
        x = sub_df.index.to_numpy().reshape(-1, 1)
        y = sub_df
        linear_model = LinearRegression().fit(x, y)
        pred = linear_model.predict(x)
        trends.append(
            {
                "transect_id": transect_id,
                "trend": linear_model.coef_[0],
                "intercept": linear_model.intercept_,
                "n_points": len(df[transect_id]),
                "n_points_nonan": len(sub_df),
                "r2_score": r2_score(y, pred),
                "mae": mean_absolute_error(y, pred),
                "mse": mean_squared_error(y, pred),
                "rmse": root_mean_squared_error(y, pred),
            }
        )
    return pd.DataFrame(trends)


# trends = get_trends(sar_files.iloc[-1]).set_index("transect_id")
trends = pd.concat(process_map(get_trends, my_files)).set_index("transect_id")
len(trends)

  0%|          | 0/2 [00:00<?, ?it/s]

EmptyDataError: No columns to parse from file

In [10]:
trends[trends.n_points_nonan > 10].sort_values("r2_score")

NameError: name 'trends' is not defined

In [11]:
trends[trends.index.str.startswith("ber")]

NameError: name 'trends' is not defined

In [12]:
trends.describe()

NameError: name 'trends' is not defined

In [13]:
(transects.trend - trends.trend).describe()

NameError: name 'trends' is not defined

In [14]:
transects.update(trends.drop_duplicates())

NameError: name 'trends' is not defined

In [15]:
transects

,site_id,orientation,along_dist,along_dist_norm,beach_slope,cil,ciu,trend,n_points,n_points_nonan,r2_score,mae,mse,rmse,intercept,ERODIBILITY,geometry
id,,,,,,,,,,,,,,,,,
aus0001-0000,aus0001,104.347648,0.000000,0.000000,0.085,0.0545,0.2000,-1.441081,767.0,428.0,0.168420,28.102591,1263.560863,35.546601,179.085729,None,"LINESTRING (153.26555 -24.7007, 153.26938 -24...."
aus0001-0001,aus0001,93.495734,98.408334,0.002935,0.050,0.0387,0.0640,-1.037105,767.0,569.0,0.097874,25.419324,1033.770813,32.152306,212.247788,None,"LINESTRING (153.26525 -24.7019, 153.2692 -24.7..."
aus0001-0002,aus0001,82.069341,198.408334,0.005918,0.050,0.0428,0.0647,-0.680019,767.0,588.0,0.053927,22.632907,838.007507,28.948359,205.106151,None,"LINESTRING (153.26539 -24.70316, 153.26931 -24..."
aus0001-0003,aus0001,81.192757,298.402523,0.008900,0.055,0.0480,0.0659,-0.405198,767.0,598.0,0.023412,20.749758,698.653187,26.432048,191.745881,None,"LINESTRING (153.26555 -24.70408, 153.26945 -24..."
aus0001-0004,aus0001,81.065473,398.402523,0.011882,0.075,0.0614,0.0922,-0.090025,767.0,608.0,0.001277,19.889328,655.810616,25.608800,175.092121,None,"LINESTRING (153.2657 -24.70497, 153.26961 -24...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ber0002-0009,ber0002,NaN,NaN,NaN,0.080,0.0638,0.1065,0.159612,199.0,199.0,0.041461,4.326802,33.378983,5.777455,127.283966,None,"LINESTRING (-64.82204 32.25336, -64.82017 32.2..."
ber0002-0010,ber0002,NaN,NaN,NaN,0.085,0.0673,0.1096,0.071946,199.0,197.0,0.010730,4.357300,26.732016,5.170301,128.858980,None,"LINESTRING (-64.82143 32.25361, -64.82029 32.2..."
ber0002-0011,ber0002,NaN,NaN,NaN,0.105,0.0797,0.1462,0.081426,199.0,198.0,0.011823,4.779105,31.534469,5.615556,129.347401,None,"LINESTRING (-64.82118 32.25369, -64.82004 32.2..."


In [16]:
trends.columns, transects.columns, trends.columns.isin(transects.columns)

NameError: name 'trends' is not defined

In [17]:
transects = transects.join(trends.loc[:, ~trends.columns.isin(transects.columns)])
transects

NameError: name 'trends' is not defined

In [18]:
transects[transects.site_id.str.startswith("sar") & ~transects.trend.isna()]

,site_id,orientation,along_dist,along_dist_norm,beach_slope,cil,ciu,trend,n_points,n_points_nonan,r2_score,mae,mse,rmse,intercept,ERODIBILITY,geometry
id,,,,,,,,,,,,,,,,,
sar0001-0000,sar0001,NaN,NaN,NaN,NaN,NaN,NaN,-0.209107,671.0,667.0,0.003085,34.337637,2024.899412,44.998882,135.992639,Medium,"LINESTRING (8.40852 38.86175, 8.40882 38.86535)"
sar0001-0001,sar0001,NaN,NaN,NaN,NaN,NaN,NaN,-0.168722,671.0,667.0,0.008700,11.931958,460.730301,21.464629,197.877604,Medium,"LINESTRING (8.4084 38.86162, 8.41092 38.86464)"
sar0001-0002,sar0001,NaN,NaN,NaN,NaN,NaN,NaN,-0.012318,671.0,669.0,0.000413,5.801157,51.994338,7.210710,206.410952,Medium,"LINESTRING (8.40893 38.86153, 8.41236 38.86393)"
sar0001-0003,sar0001,NaN,NaN,NaN,NaN,NaN,NaN,-0.043548,671.0,669.0,0.016188,3.202754,16.331152,4.041182,239.500879,Medium,"LINESTRING (8.40904 38.86129, 8.41333 38.8626)"
sar0001-0004,sar0001,NaN,NaN,NaN,NaN,NaN,NaN,0.011825,671.0,669.0,0.001501,2.692829,13.178444,3.630213,258.136097,Medium,"LINESTRING (8.40904 38.86176, 8.4133 38.86037)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
sar2541-0000,sar2541,NaN,NaN,NaN,NaN,NaN,NaN,-0.297976,1371.0,1371.0,0.203601,5.662483,50.986351,7.140473,199.127124,High,"LINESTRING (8.85399 38.88006, 8.85388 38.87736)"
sar2541-0001,sar2541,NaN,NaN,NaN,NaN,NaN,NaN,0.011742,1371.0,1371.0,0.000671,3.998349,30.150238,5.490923,165.933848,High,"LINESTRING (8.85428 38.88005, 8.85417 38.87735)"
sar2541-0002,sar2541,NaN,NaN,NaN,NaN,NaN,NaN,0.048766,1371.0,1371.0,0.008370,4.829729,41.363000,6.431407,161.418975,High,"LINESTRING (8.85502 38.87993, 8.85398 38.87735)"


In [19]:
transects.drop_duplicates().to_file("transects_extended.geojson", driver="GeoJSON")